#Preprocessing & Ekstraksi Fitur

In [38]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import RandomOverSampler
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Load Dataset
df = pd.read_csv('dataset_ruangguru.csv')

# 2. Text Cleaning
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['content'].apply(clean_text)

# Menghapus baris yang menjadi kosong setelah dibersihkan
df = df[df['clean_text'] != '']

# 3. Label Encoding (0 = Negatif, 1 = Netral, 2 = Positif)
le = LabelEncoder()
df['encoded_label'] = le.fit_transform(df['label'])

# 4. Penanganan Imbalanced Data
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(df[['clean_text']], df['encoded_label'])

# 5. Tokenisasi & Padding
vocab_size = 5000
max_length = 100
trunc_type = 'post'
padding_type = 'pre'
oov_tok = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(X_resampled['clean_text'])

X_sequences = tokenizer.texts_to_sequences(X_resampled['clean_text'])
X_padded = pad_sequences(X_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

# 6. Pembagian Split Data
# Skema 1 & 2 (80/20)
X_train_80, X_test_20, y_train_80, y_test_20 = train_test_split(
    X_padded, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)
# Skema 3 (90/10)
X_train_90, X_test_10, y_train_90, y_test_10 = train_test_split(
    X_padded, y_resampled, test_size=0.1, random_state=42, stratify=y_resampled
)

print("=== PERSIAPAN DATA FINAL SELESAI ===")
print("Dimensi Latih 80/20:", X_train_80.shape)
print("Dimensi Latih 90/10:", X_train_90.shape)

=== PERSIAPAN DATA FINAL SELESAI ===
Dimensi Latih 80/20: (18628, 100)
Dimensi Latih 90/10: (20957, 100)


Skema 1 - Algoritma LSTM (Split 80/20)

In [40]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("=== MELATIH SKEMA 1: LSTM (80/20) ===")

model_lstm_80 = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64),
    LSTM(64, return_sequences=False),
    Dropout(0.5), # Regularisasi untuk mencegah overfitting
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model_lstm_80.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)

history_lstm_80 = model_lstm_80.fit(
    X_train_80, y_train_80,
    validation_data=(X_test_20, y_test_20),
    epochs=15,
    batch_size=128,
    callbacks=[early_stop]
)

loss_1, acc_1 = model_lstm_80.evaluate(X_test_20, y_test_20, verbose=0)
print(f"\n=> Akurasi Skema 1: {acc_1 * 100:.2f}%")

=== MELATIH SKEMA 1: LSTM (80/20) ===
Epoch 1/15
146/146 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6359 - loss: 0.7758 - val_accuracy: 0.8089 - val_loss: 0.4847
Epoch 2/15
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.8924 - loss: 0.3257 - val_accuracy: 0.9227 - val_loss: 0.2426
Epoch 3/15
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9410 - loss: 0.1910 - val_accuracy: 0.9410 - val_loss: 0.2042
Epoch 4/15
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9554 - loss: 0.1474 - val_accuracy: 0.9457 - val_loss: 0.1905
Epoch 5/15
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9626 - loss: 0.1206 - val_accuracy: 0.9465 - val_loss: 0.1694
Epoch 6/15
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9684 - loss: 0.1008 - val_accuracy: 0.9577 - val_loss: 0.1528
Epoch 7/15
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9699 - loss: 0.0976 - val_accuracy: 0.9568 - val_loss: 0.1562
Epoch 8/15
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accu

Skema 2 - Algoritma GRU (Split 80/20)

In [41]:
from tensorflow.keras.layers import GRU

print("=== MELATIH SKEMA 2: GRU (80/20) ===")

model_gru_80 = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64),
    GRU(64, return_sequences=False),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model_gru_80.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history_gru_80 = model_gru_80.fit(
    X_train_80, y_train_80,
    validation_data=(X_test_20, y_test_20),
    epochs=15,
    batch_size=128,
    callbacks=[early_stop]
)

loss_2, acc_2 = model_gru_80.evaluate(X_test_20, y_test_20, verbose=0)
print(f"\n=> Akurasi Skema 2: {acc_2 * 100:.2f}%")

=== MELATIH SKEMA 2: GRU (80/20) ===
Epoch 1/15
146/146 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.6442 - loss: 0.7726 - val_accuracy: 0.8377 - val_loss: 0.4626
Epoch 2/15
146/146 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8962 - loss: 0.3056 - val_accuracy: 0.9279 - val_loss: 0.2314
Epoch 3/15
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9455 - loss: 0.1787 - val_accuracy: 0.9414 - val_loss: 0.1920

=> Akurasi Skema 2: 83.77%


Skema 3 - Algoritma LSTM (Split 90/10)

In [42]:
print("=== MELATIH SKEMA 3: LSTM (90/10) ===")

model_lstm_90 = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model_lstm_90.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history_lstm_90 = model_lstm_90.fit(
    X_train_90, y_train_90,
    validation_data=(X_test_10, y_test_10),
    epochs=15,
    batch_size=128,
    callbacks=[early_stop]
)

loss_3, acc_3 = model_lstm_90.evaluate(X_test_10, y_test_10, verbose=0)
print(f"\n=> Akurasi Skema 3: {acc_3 * 100:.2f}%")

=== MELATIH SKEMA 3: LSTM (90/10) ===
Epoch 1/15
164/164 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.6719 - loss: 0.7263 - val_accuracy: 0.8424 - val_loss: 0.4072
Epoch 2/15
164/164 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9130 - loss: 0.2714 - val_accuracy: 0.9309 - val_loss: 0.2137
Epoch 3/15
164/164 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9469 - loss: 0.1713 - val_accuracy: 0.9498 - val_loss: 0.1664

=> Akurasi Skema 3: 84.24%


#Pengujian

In [ ]:
print("=== PENGUJIAN INFERENCE ===")

# Kumpulan teks baru yang belum pernah dilihat model
teks_baru = [
    "aplikasi ini sangat membantu saya belajar untuk UTBK, videonya jelas dan mudah dipahami. mantap ruangguru!",
    "sejak update kemarin aplikasinya sering force close, video buffering terus padahal sinyal bagus.",
    "cukup bagus untuk belajar harian, walau kadang ada soal yang kurang lengkap pembahasannya."
]

# 1. Bersihkan Teks
teks_bersih = [clean_text(t) for t in teks_baru]

# 2. Tokenisasi & Padding
sekuens_baru = tokenizer.texts_to_sequences(teks_bersih)
padded_baru = pad_sequences(sekuens_baru, maxlen=max_length, padding=padding_type, truncating=trunc_type)

# 3. Prediksi menggunakan model Skema 1
prediksi_probabilitas = model_lstm_80.predict(padded_baru)
prediksi_kelas = np.argmax(prediksi_probabilitas, axis=1)

# 4. Konversi kembali ke label teks (0: Negatif, 1: Netral, 2: Positif)
label_aktual = le.inverse_transform(prediksi_kelas)

# Tampilkan Hasil
for i in range(len(teks_baru)):
    print(f"Teks Input : {teks_baru[i]}")
    print(f"Prediksi   : {label_aktual[i]}")
    print("-" * 50)

=== PENGUJIAN INFERENCE ===
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Teks Input : aplikasi ini sangat membantu saya belajar untuk UTBK, videonya jelas dan mudah dipahami. mantap ruangguru!
Prediksi   : Positif
--------------------------------------------------
Teks Input : sejak update kemarin aplikasinya sering force close, video buffering terus padahal sinyal bagus.
Prediksi   : Negatif
--------------------------------------------------
Teks Input : cukup bagus untuk belajar harian, walau kadang ada soal yang kurang lengkap pembahasannya.
Prediksi   : Positif
--------------------------------------------------
